<a href="https://colab.research.google.com/github/Lord-LLM/Zero-Shot-Job-Resume-to-Description-Matcher/blob/main/Zero_Shot_Job_Resume_to_Description_Matcher.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install kagglehub

In [2]:
import kagglehub

In [20]:
kagglehub.login()

Kaggle credentials set.
Kaggle credentials successfully validated.


In [21]:
from kagglehub import KaggleDatasetAdapter

In [24]:
df_resume = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "snehaanbhawal/resume-dataset",
    "Resume/Resume.csv",
)

Using Colab cache for faster access to the 'resume-dataset' dataset.


In [26]:
df_resume.head()

,ID,Resume_str,Resume_html,Category
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR
2,33176873,HR DIRECTOR Summary Over 2...,"<div class=""fontsize fontface vmargins hmargin...",HR
3,27018550,HR SPECIALIST Summary Dedica...,"<div class=""fontsize fontface vmargins hmargin...",HR
4,17812897,HR MANAGER Skill Highlights ...,"<div class=""fontsize fontface vmargins hmargin...",HR


In [7]:
!pip install datasets

In [8]:
from datasets import load_dataset

dataset = load_dataset("azrai99/job-dataset", split="train")

README.md:   0%|          | 0.00/5.20k [00:00<?, ?B/s]

jobstreet_all_job_dataset.csv: reconstructing file:   0%|          |  0.00B /  123MB            

jobstreet_all_job_dataset.csv: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/59306 [00:00<?, ? examples/s]

In [13]:
import pandas as pd
pd.set_option('display.max_columns', None)

df_jobPostings = pd.DataFrame(dataset)

print(f"Dataset Shape: {df_jobPostings.shape} (Rows, Columns)")
df_jobPostings.head()

Dataset Shape: (59306, 11) (Rows, Columns)


,job_id,job_title,company,descriptions,location,category,subcategory,role,type,salary,listingDate
0,74630583,Procurement Executive (Contract),Coca-Cola Bottlers (Malaysia) Sdn Bhd,Position Purpose\nManage aspects of procuremen...,Negeri Sembilan,"Manufacturing, Transport & Logistics","Purchasing, Procurement & Inventory",procurement-executive,Contract/Temp,None,2024-03-21T05:58:35Z
1,74660602,Account Executive/ Assistant,Acoustic & Lighting System Sdn Bhd,We are looking for a Account Executive/ Assist...,Petaling,Accounting,Bookkeeping & Small Practice Accounting,executive-assistant,Full time,"RM 2,800 – RM 3,200 per month",2024-03-22T06:52:57Z
2,74655679,"Data Analyst - Asset Management, SPX Express",Shopee Mobile Malaysia Sdn Bhd,Performs detailed data analysis on existing sp...,Klang District,"Manufacturing, Transport & Logistics",Analysis & Reporting,asset-management-analyst,Full time,None,2024-03-22T04:22:43Z
3,74657624,Service Engineer,Sun Medical Systems Sdn Bhd,"You are important for troubleshooting, install...",Petaling,Engineering,Electrical/Electronic Engineering,services-engineer,Full time,"RM 3,000 – RM 3,500 per month",2024-03-22T05:32:09Z
4,74679363,Purchasing Executive,Magnet Security & Automation Sdn. Bhd.,"MAG is a trailblazer in the industry, boasting...",Hulu Langat,"Manufacturing, Transport & Logistics","Purchasing, Procurement & Inventory",purchasing-executive,Full time,"RM 2,800 – RM 3,500 per month",2024-03-23T03:56:39Z


In [15]:
df_resume.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 962 entries, 0 to 961
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Category  962 non-null    object
 1   Resume    962 non-null    object
dtypes: object(2)
memory usage: 15.2+ KB


In [16]:
df_resume.describe()

,Category,Resume
count,962,962
unique,25,166
top,Java Developer,"Technical Skills Web Technologies: Angular JS,..."
freq,84,18


In [17]:
df_jobPostings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 59306 entries, 0 to 59305
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   job_id        59306 non-null  int64 
 1   job_title     59306 non-null  object
 2   company       59306 non-null  object
 3   descriptions  59306 non-null  object
 4   location      59306 non-null  object
 5   category      59306 non-null  object
 6   subcategory   59306 non-null  object
 7   role          57370 non-null  object
 8   type          59306 non-null  object
 9   salary        26641 non-null  object
 10  listingDate   59306 non-null  object
dtypes: int64(1), object(10)
memory usage: 5.0+ MB


In [18]:
df_jobPostings.describe()

,job_id
count,5.930600e+04
mean,7.497790e+07
std,9.319422e+05
min,7.134223e+07
25%,7.433293e+07
50%,7.478895e+07
75%,7.572671e+07
max,7.669411e+07


In [28]:
import re
def clean_text(text: str) -> str:
    """
    Minimal cleaning: normalize whitespace, strip stray page-number/OCR
    artifacts. Deliberately NOT lowercasing, stemming, or removing
    punctuation — sentence-transformer models are trained on natural
    sentences and are robust to case/punctuation; over-cleaning throws
    away information the model actually uses.
    """
    if not isinstance(text, str):
        return ""

    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"page\s*\d+\s*(of\s*\d+)?", "", text, flags=re.IGNORECASE)

    # Strip leading/trailing whitespace
    return text.strip()



# Regex-based section splitter

SECTION_HEADERS = [
    "summary", "objective", "skills", "technical skills", "core competencies",
    "experience", "work experience", "professional experience", "employment history",
    "education", "certifications", "projects", "achievements",
]

_headers_sorted = sorted(SECTION_HEADERS, key=len, reverse=True)
_header_pattern = re.compile(
    r"(?<![A-Za-z])(" + "|".join(re.escape(h) for h in _headers_sorted) + r")(?![A-Za-z])",
    flags=re.IGNORECASE,
)


def split_sections(text: str) -> dict:
    """
    Find every header occurrence in `text` and slice the text between
    consecutive headers. Returns a dict of {header_name: section_text}.
    """
    matches = list(_header_pattern.finditer(text))
    sections = {}

    for i, match in enumerate(matches):
        header = match.group(1).lower()
        start = match.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        section_text = text[start:end].strip(" :\u2022-\t")
        if section_text:
            # If the same header appears twice, append rather than overwrite
            sections[header] = (sections.get(header, "") + " " + section_text).strip()

    return sections


def get_skills_experience_summary(text: str) -> str:
    """
    Convenience wrapper for pulling out just the skills + experience sections and concatenates them into one string,
    ready to embed. Falls back to the full cleaned text if no headers were
    found (some resumes won't match any keyword — don't silently produce
    an empty string).
    """
    sections = split_sections(text)

    wanted = [
        sections.get("skills", ""),
        sections.get("technical skills", ""),
        sections.get("core competencies", ""),
        sections.get("experience", ""),
        sections.get("work experience", ""),
        sections.get("professional experience", ""),
        sections.get("employment history", ""),
    ]
    combined = " ".join(w for w in wanted if w).strip()

    return combined if combined else text



if __name__ == "__main__":

     df_resume["clean_text"] = df_resume["Resume_str"].apply(clean_text)
     df_resume["embed_input_naive"] = df_resume["clean_text"]

     df_resume["embed_input_structured"] = df_resume["clean_text"].apply(get_skills_experience_summary)

     for i in range(3):
         print(df_resume.loc[i, "Category"])
         print(split_sections(df_resume.loc[i, "clean_text"]).keys())

HR
dict_keys(['summary', 'experience', 'skills', 'education'])
HR
dict_keys(['summary', 'experience', 'education', 'skills'])
HR
dict_keys(['summary', 'experience', 'skills', 'education'])
